# Séries Temporais: da Exploração à Previsão

Séries temporais estão presentes em quase todo sistema que um desenvolvedor web mantém: métricas de acesso, volume de pedidos, uso de CPU, fila de mensagens, receita diária. A diferença fundamental em relação a dados tabulares comuns é que **as observações não são independentes** — o valor de agora depende do que aconteceu antes. Isso muda tudo: como limpamos, como criamos features, como dividimos treino e teste, e como avaliamos.

## O que faremos nesta aula

Vamos percorrer o pipeline completo de ciência de dados para séries temporais, do dado bruto até dois modelos preditivos treinados e avaliados corretamente:

- **Entendimento e definição do problema:** identificar o que a série representa e qual é o objetivo — prever um valor futuro (regressão) ou classificar um evento futuro (classificação).
- **Exploração e limpeza:** carregar o dataset, converter datas, ordenar por tempo, visualizar a série e diagnosticar buracos e anomalias.
- **Tratamento de valores ausentes:** aplicar técnicas próprias de dados temporais (forward fill, interpolação por tempo, imputação sazonal), que respeitam a ordem das observações.
- **Engenharia de features temporais:** extrair informação do relógio e do calendário (hora, dia da semana, feriado), da própria série (lags e janelas móveis) e da sazonalidade (termos de Fourier).
- **Análise de sazonalidade e tendência:** confirmar visualmente e com ACF/PACF os padrões que justificam as features criadas.
- **Divisão treino/validação/teste no tempo:** separar respeitando a ordem cronológica, sem nunca embaralhar, para não vazar informação do futuro.
- **Modelagem preditiva:** treinar regressão (prever a demanda da próxima hora) e classificação (prever se a próxima hora é horário de pico), com `Pipeline` do scikit-learn.
- **Avaliação e serialização:** medir desempenho com métricas adequadas e salvar o modelo pronto para servir atrás de um endpoint.

> **O fio condutor da aula é o *look-ahead bias* (viés de olhar o futuro).** Praticamente todo erro sério em séries temporais é uma variação de usar, no treino, informação que só estaria disponível depois. Vamos marcar cada ponto onde esse risco aparece.

# 1. Carregamento e Visualização Inicial dos Dados

**Dataset:** usaremos o *Bike Sharing Dataset* (UCI Machine Learning Repository), com a contagem **horária** de aluguéis de bicicletas em Washington D.C. entre 2011 e 2012, junto de variáveis explicativas (temperatura, umidade, velocidade do vento, feriado, dia da semana, condição do tempo). É uma série temporal multivariada com sazonalidade diária, semanal e anual bem marcada — perfeita para o que vamos estudar.

A célula abaixo baixa o arquivo horário direto de um espelho no GitHub. Se a rede estiver bloqueada, ela cai automaticamente para uma **série sintética** com a mesma estrutura (mesmas colunas, mesma sazonalidade e alguns buracos propositais), de modo que o restante do notebook roda de ponta a ponta em qualquer ambiente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RNG = np.random.default_rng(42)

def carregar_bike_horario():
    "Tenta baixar o hour.csv do UCI Bike Sharing; se falhar, gera equivalente sintetico."
    url = ("https://raw.githubusercontent.com/muditp19/"
           "UCI_Bike-sharing-dataset/master/hour.csv")
    try:
        raw = pd.read_csv(url)
        raw["datetime"] = (pd.to_datetime(raw["dteday"])
                           + pd.to_timedelta(raw["hr"], unit="h"))
        # renomeia para nomes mais legíveis, mantendo o essencial
        raw = raw.rename(columns={
            "cnt": "count", "hum": "humidity", "weathersit": "weather",
            "workingday": "workingday", "temp": "temp",
            "windspeed": "windspeed", "holiday": "holiday", "season": "season",
        })
        cols = ["datetime", "count", "temp", "humidity", "windspeed",
                "holiday", "workingday", "weather", "season"]
        df = raw[cols].sort_values("datetime").set_index("datetime")
        print(f"[OK] Dataset real carregado da web: {len(df)} registros.")
        return df
    except Exception as e:
        print(f"[FALLBACK] Sem acesso à web ({type(e).__name__}). Gerando série sintética.")
        idx = pd.date_range("2011-01-01", "2012-12-31 23:00", freq="h")
        h = idx.hour.values; wd = idx.dayofweek.values; doy = idx.dayofyear.values
        # dois picos diários em dia útil, curva única no fim de semana
        base = (60
                + 180*np.exp(-((h-8)**2)/6) + 200*np.exp(-((h-18)**2)/6))
        weekend = 120*np.exp(-((h-14)**2)/20)
        util = wd < 5
        nivel = np.where(util, base, weekend)
        sazonal_ano = 1 + 0.5*np.sin(2*np.pi*(doy-80)/365)   # verão alto
        ruido = RNG.normal(0, 15, size=len(idx))
        count = np.clip((nivel*sazonal_ano + ruido), 1, None).round().astype(int)
        df = pd.DataFrame({
            "count": count,
            "temp": 0.5 + 0.3*np.sin(2*np.pi*(doy-80)/365) + RNG.normal(0,0.03,len(idx)),
            "humidity": np.clip(0.6 + RNG.normal(0,0.15,len(idx)), 0, 1),
            "windspeed": np.clip(RNG.gamma(2,0.08,len(idx)), 0, 1),
            "holiday": 0,
            "workingday": util.astype(int),
            "weather": RNG.integers(1,4,len(idx)),
            "season": ((idx.month % 12)//3 + 1),
        }, index=idx)
        # abre ~160 buracos aleatórios para o exercício de imputação
        drop = RNG.choice(len(df), size=160, replace=False)
        df = df.drop(df.index[drop])
        return df

df = carregar_bike_horario()
print(df.head())
print(df.dtypes)
print(f"Período: {df.index.min()} até {df.index.max()}  |  registros: {len(df)}")


**Diagnóstico de continuidade — a primeira coisa a checar numa série temporal.** O índice está ordenado no tempo, mas será que temos *todas* as horas do período, sem buracos? Muita gente pula essa verificação e paga caro depois: features de lag e de janela móvel assumem que a linha anterior é, de fato, a hora anterior. Se houver saltos, `lag24` deixa de ser "ontem no mesmo horário" e vira lixo silencioso.

Comparamos o número de registros com o número de horas que *deveriam* existir entre o início e o fim:

In [ ]:
horas_esperadas = pd.date_range(df.index.min(), df.index.max(), freq="h")
faltando = len(horas_esperadas) - len(df)
print(f"Horas esperadas no período: {len(horas_esperadas)}")
print(f"Horas presentes no dataset: {len(df)}")
print(f"Horas ausentes (buracos):   {faltando}")

# Reindexa para a grade horária COMPLETA: as horas ausentes viram linhas com NaN.
# Só assim os lags e as janelas móveis ficam temporalmente corretos.
df = df.reindex(horas_esperadas)
df.index.name = "datetime"
print(f"\nApós reindexar: {len(df)} linhas; NaN em 'count': {df['count'].isna().sum()}")


Agora `df` tem uma linha para cada hora do período. As horas que faltavam existem como `NaN` — vamos tratá-las na seção de limpeza. Esse é o momento em que a série "vira" verdadeiramente temporal: índice regular, frequência fixa, buracos explícitos.

**Visualização inicial.** Plotamos a contagem de aluguéis ao longo do tempo. Com quase dois anos de dados horários, o gráfico completo fica denso, então também mostramos um recorte de duas semanas para enxergar a forma diária.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6))

df["count"].plot(ax=ax1, lw=0.4)
ax1.set(title="Aluguéis de bicicleta — série horária completa (2011–2012)",
        xlabel="Data", ylabel="Aluguéis/hora")

recorte = df.loc["2011-06-06":"2011-06-19", "count"]
recorte.plot(ax=ax2, marker=".", ms=3, lw=0.8)
ax2.set(title="Recorte de duas semanas — repare no duplo pico diário em dias úteis",
        xlabel="Data", ylabel="Aluguéis/hora")
plt.tight_layout(); plt.show()


**O que esperar.** Na série completa aparecem a sazonalidade anual (mais uso no verão, menos no inverno) e uma leve tendência de crescimento de 2011 para 2012. No recorte de duas semanas surge a assinatura diária: em dias úteis, dois picos (manhã ~8h e fim de tarde ~18h, o *commute*); nos fins de semana, uma curva única e mais baixa, espalhada pela tarde. Também temos variáveis como `season` e `weather`, categóricas, que servirão de features.

**Exercício 1.** Identifique visualmente no gráfico períodos de alta e baixa demanda. Há padrão distinto entre fins de semana e dias úteis? *(Dica: sobreponha curvas separando por tipo de dia, ou agregue a média por hora e por dia da semana — é exatamente o que faremos a seguir.)*

In [ ]:
df["weekday"] = df.index.dayofweek   # 0 = segunda ... 6 = domingo
df["hour"] = df.index.hour
media_semana = (df.dropna(subset=["count"])
                  .groupby(["weekday", "hour"])["count"].mean()
                  .reset_index())

plt.figure(figsize=(11, 4))
sns.lineplot(data=media_semana, x="hour", y="count",
             hue="weekday", palette="tab10")
plt.title("Demanda média por hora do dia, para cada dia da semana")
plt.xlabel("Hora do dia"); plt.ylabel("Aluguéis médios")
plt.legend(title="Dia", labels=["Seg","Ter","Qua","Qui","Sex","Sáb","Dom"])
plt.tight_layout(); plt.show()


Essa visão confirma a intuição: de segunda a sexta há picos agudos de manhã e no fim da tarde (deslocamento casa–trabalho); sábado e domingo têm um único platô à tarde. Essa diferença é forte o suficiente para que `hour` e `weekday` sejam features valiosas — e já sugere a tarefa de classificação que definiremos depois: *distinguir horário de pico*.

# 2. Limpeza de Dados e Tratamento de Valores Ausentes

Na limpeza cuidamos de dois problemas: **dados faltantes** (os buracos que a reindexação tornou explícitos) e **valores anômalos** (outliers).

## 2.1 Identificação de dados faltantes

Já sabemos, pela reindexação, que existem horas sem registro. Vamos quantificar por coluna e visualizar *onde* no tempo os buracos se concentram — isso indica se a ausência é aleatória (falha pontual de sensor) ou sistemática (um período inteiro fora do ar), o que muda a estratégia de imputação.

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum())

falt = df["count"].isna()
plt.figure(figsize=(11, 2.5))
df["count"].plot(lw=0.4, label="Disponível")
plt.scatter(df.index[falt], np.zeros(falt.sum()),
            color="red", s=8, label="Faltante", zorder=3)
plt.title("Série com valores faltantes destacados (marcadores vermelhos na base)")
plt.legend(loc="upper right"); plt.tight_layout(); plt.show()


Os pontos vermelhos são as horas ausentes. Como estão espalhados e não formam blocos longos, temos boas condições para imputar sem distorcer a série — para lacunas curtas, os vizinhos temporais são bons palpites.

## 2.2 Técnicas de imputação para séries temporais

Diferentemente de dados independentes, aqui devemos **respeitar a ordem e a dependência temporal** ao preencher. As técnicas mais usadas:

- **Remoção (deletion):** descartar as linhas com ausência. Só é aceitável se os buracos forem poucos e aleatórios; em séries longas com lacunas frequentes, joga fora informação demais e ainda reintroduz a descontinuidade que a reindexação resolveu.
- **Forward/backward fill:** repete o último valor conhecido (`ffill`) ou o próximo (`bfill`). Simples e útil para lacunas curtas, mas assume que a série ficou estável durante o buraco — ruim para gaps longos.
- **Interpolação:** estima valores intermediários a partir dos vizinhos (linear, spline ou por tempo). Com `method="time"`, o pandas pondera pela distância temporal real, o que é o certo quando o índice pode ter espaçamento irregular. Boa quando a série varia de forma suave.
- **Janela local (média/mediana móvel):** preenche com uma estatística da vizinhança recente — por exemplo, a média das mesmas horas em dias próximos.
- **Imputação sazonal:** se há sazonalidade forte, usa valores de ciclos anteriores (ex.: a mesma hora do mesmo dia da semana).
- **Imputação por modelo:** treina um modelo (ARIMA ou supervisionado) para prever o valor ausente — mais poderoso e mais caro.

> **Atenção ao `method="ffill"` como argumento de `fillna`.** A forma `df.fillna(method="ffill")` foi **removida no pandas 3.0**. Use os métodos dedicados `df["col"].ffill()` e `df["col"].bfill()`.

In [ ]:
# Variáveis contínuas meteorológicas: interpolação por tempo (suave).
for col in ["temp", "humidity", "windspeed"]:
    df[col] = df[col].interpolate(method="time")

# Colunas categóricas/calendário: forward fill (o clima/estação da hora anterior
# é o melhor palpite para uma lacuna curta).
for col in ["holiday", "workingday", "weather", "season"]:
    df[col] = df[col].ffill().bfill()

# Alvo (count): forward fill para lacunas curtas. Guardamos ANTES uma máscara
# das horas que eram originalmente ausentes — será útil para não avaliar o
# modelo em cima de valores que nós mesmos inventamos.
df["count_imputado"] = df["count"].isna()
df["count"] = df["count"].interpolate(method="time").round()

print("Ausentes remanescentes por coluna:")
print(df.isna().sum())


**Exercício 2.** Compare métodos de imputação para `count`. Por exemplo, confronte a interpolação por tempo com o preenchimento pela média histórica daquela hora-do-dia/dia-da-semana. Avalie visualmente: o resultado é plausível ou introduz descontinuidades? *(Dica: uma boa validação é ocultar artificialmente valores que existem, imputá-los e medir o erro contra o valor verdadeiro.)*

**Boas práticas.** Sempre documente e justifique a técnica escolhida e, se possível, meça o impacto no modelo final. Para porcentagens altas de ausência, ferramentas como o pacote `missingno` ajudam a diagnosticar se o padrão de ausência é aleatório ou sistemático.

## 2.3 Detecção e tratamento de outliers

Além de faltantes, vale checar valores anômalos: picos ou quedas abruptas fora do padrão (por exemplo, uma contagem muito acima de qualquer dia típico, possível erro de registro). Podemos detectar por regra estatística (fora de ±3 desvios, ou fora do intervalo interquartil) e por inspeção visual. Se um ponto for comprovadamente errôneo, as opções são remover, corrigir com fonte externa, ou tratar como faltante e imputar.

In [ ]:
q1, q3 = df["count"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 3 * iqr
suspeitos = df["count"] > lim_sup
print(f"Limite superior (Q3 + 3·IQR): {lim_sup:.0f}")
print(f"Horas acima do limite: {suspeitos.sum()} "
      f"({100*suspeitos.mean():.2f}% da série)")

plt.figure(figsize=(11, 3))
df.boxplot(column="count", by="hour", ax=plt.gca())
plt.suptitle(""); plt.title("Distribuição de 'count' por hora do dia (boxplot)")
plt.xlabel("Hora do dia"); plt.ylabel("Aluguéis"); plt.tight_layout(); plt.show()


O boxplot por hora mostra que os valores altos se concentram justamente nas horas de pico — ou seja, **não são erros, são a sazonalidade real**. Isso é um lembrete importante: em séries temporais, um "outlier global" muitas vezes é apenas um valor normal *para aquele contexto temporal*. Por isso não removeremos nada aqui.

**Exercício 3.** Investigue se há algum outlier genuíno em `count` que não seja explicado pela hora do dia. *(Dica: calcule o desvio de cada ponto em relação à média da sua própria hora-do-dia e dia-da-semana; anomalias reais aparecem como desvios grandes mesmo após esse ajuste.)*

# 3. Engenharia de Features em Séries Temporais

Com os dados limpos, criamos atributos que representem os padrões temporais. O sucesso de modelos de ML em séries temporais depende fortemente de como transformamos o tempo e a própria história da série em colunas úteis. Aqui entram conhecimento de domínio e cuidado com vazamento.

## 3.1 Componentes de data/hora

Data e hora são fontes diretas de sazonalidade e efeitos de calendário. A partir do índice extraímos:

- **Hora do dia** (`hour`) — o principal driver da demanda.
- **Dia da semana** (`weekday`) — separa dias úteis de fins de semana.
- **Mês** (`month`) — sazonalidade anual.
- **Ano** (`year`) — captura a tendência de crescimento entre 2011 e 2012.
- **Fim de semana** (`is_weekend`) — derivado de `weekday`.
- **Feriado / dia útil** — já vêm no dataset (`holiday`, `workingday`).

**Codificação cíclica.** Hora e dia da semana são *circulares*: a hora 23 é vizinha da hora 0, mas para o modelo `23` e `0` parecem distantes. Para modelos lineares e redes neurais, representamos essas variáveis com **seno e cosseno**, de modo que o fim do ciclo encoste no começo. Modelos de árvore não precisam disso (fazem splits), mas a codificação cíclica não os prejudica.

In [ ]:
df["month"] = df.index.month
df["year"] = df.index.year
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# Codificação cíclica de hora (período 24) e dia da semana (período 7)
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["wday_sin"] = np.sin(2*np.pi*df["weekday"]/7)
df["wday_cos"] = np.cos(2*np.pi*df["weekday"]/7)

print(df[["hour","hour_sin","hour_cos","weekday","is_weekend"]].head())


## 3.2 Lag features (defasagens)

As features mais características de séries temporais são os **lags**: valores passados da própria série usados como preditores. Eles inserem explicitamente a dependência temporal no conjunto de treino. Para prever a demanda em `t+1`, é útil saber quantas bikes foram alugadas na última hora (`t`), há 24 horas (mesma hora de ontem) e há 168 horas (mesma hora, mesma semana passada):

- **lag de 1–2 horas** captura inércia imediata;
- **lag de 24 horas** captura o ciclo diário;
- **lag de 168 horas** captura o ciclo semanal.

Como a série foi reindexada para uma grade horária completa, `shift(24)` agora significa *exatamente* "24 horas atrás" — foi por isso que insistimos na continuidade lá no início.

In [ ]:
df["lag1"]   = df["count"].shift(1)     # 1 hora atrás
df["lag24"]  = df["count"].shift(24)    # mesmo horário, dia anterior
df["lag168"] = df["count"].shift(168)   # mesmo horário, semana anterior
print(df[["count","lag1","lag24","lag168"]].head(3))
print(df[["count","lag1","lag24","lag168"]].tail(3))


As primeiras linhas ficam com `NaN` nessas colunas (não há passado suficiente). Vamos descartá-las na hora de montar as matrizes de treino, em vez de preencher com zero — zero seria um valor de demanda plausível e enganaria o modelo.

## 3.3 Rolling windows (janelas móveis)

Além de valores pontuais do passado, criamos **agregações do passado recente**: médias, desvios e extremos numa janela deslizante. Elas suavizam ruído e destacam a tendência local.

> **Ponto crítico de vazamento.** A janela precisa terminar em `t-1`, **nunca** incluir a hora atual `t` — senão o modelo "vê" parte da resposta. Por isso aplicamos `.shift(1)` depois do `.rolling(...)`: a janela é calculada até a hora anterior.

In [ ]:
df["roll_mean_3"]  = df["count"].rolling(3).mean().shift(1)    # média das 3h anteriores
df["roll_mean_24"] = df["count"].rolling(24).mean().shift(1)   # média do último dia
df["roll_std_24"]  = df["count"].rolling(24).std().shift(1)    # volatilidade do último dia
df["roll_max_24"]  = df["count"].rolling(24).max().shift(1)    # pico do último dia
print(df[["count","roll_mean_3","roll_mean_24","roll_std_24"]].dropna().head())


## 3.4 Features sazonais avançadas (Fourier)

Para sazonalidades múltiplas e suaves, **termos de Fourier** — pares de seno e cosseno em diferentes períodos — deixam modelos lineares capturarem periodicidade sem precisar de uma dummy por hora. Já criamos os termos diários na codificação cíclica; abaixo adicionamos um par para o ciclo **anual** (período de 365,25 dias, em horas). É o mesmo mecanismo que o Prophet usa internamente.

In [ ]:
horas_no_ano = 365.25 * 24
t = (df.index - df.index[0]).total_seconds() / 3600.0   # horas desde o início
df["ano_sin"] = np.sin(2*np.pi*t/horas_no_ano)
df["ano_cos"] = np.cos(2*np.pi*t/horas_no_ano)
print(df[["ano_sin","ano_cos"]].iloc[[0, 4380, 8760]])   # início, ~meio, ~1 ano


**Exercício 4.** Crie pelo menos três novas features temporais além das sugeridas. Ideias: um indicador `rush_hour` (hora ∈ {7,8,9,17,18,19}); lags da temperatura (`temp.shift(1)`), permitindo ao modelo perceber "está esfriando, a demanda pode cair"; ou a diferença `count(t-1) − count(t-2)` como aceleração recente. Para cada uma, escreva sua hipótese de por que seria preditiva e teste a correlação com o alvo.

# 4. Análise de Sazonalidade e Tendência

Antes de modelar, vale confirmar e quantificar os padrões — isso justifica as features criadas e antecipa quais serão importantes.

## 4.1 Tendência e sazonalidades de calendário

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

df["count"].resample("MS").mean().plot(ax=axes[0], marker="o")
axes[0].set(title="Tendência — média mensal", xlabel="", ylabel="Aluguéis médios")

df.groupby(df.index.hour)["count"].mean().plot(ax=axes[1])
axes[1].set(title="Sazonalidade diária", xlabel="Hora do dia", ylabel="")

df.groupby(df.index.month)["count"].mean().plot.bar(ax=axes[2])
axes[2].set(title="Sazonalidade anual", xlabel="Mês", ylabel="")
plt.tight_layout(); plt.show()


Esperamos ver: crescimento de 2011 para 2012 (tendência); a curva de duplo pico no perfil diário; e demanda maior nos meses quentes. Cada um desses padrões corresponde diretamente a uma família de features — `year`, `hour`/termos diários, e `month`/termos anuais.

## 4.2 Autocorrelação: ACF e PACF

A ferramenta canônica para *escolher lags com evidência* (em vez de no chute) é a **função de autocorrelação (ACF)** e sua versão parcial (**PACF**). Elas medem o quanto a série se correlaciona consigo mesma em cada defasagem. Picos em defasagens específicas dizem quais lags carregam sinal.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

serie = df["count"].dropna()
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6))
plot_acf(serie, lags=180, ax=a1)
a1.set_title("ACF — note os picos em múltiplos de 24 (diário) e em 168 (semanal)")
plot_pacf(serie, lags=72, ax=a2, method="ywm")
a2.set_title("PACF — defasagens com contribuição direta")
plt.tight_layout(); plt.show()


Os picos da ACF em 24, 48, 168 confirmam quantitativamente a sazonalidade diária e semanal — exatamente os lags que criamos (`lag1`, `lag24`, `lag168`). É a ponte entre "eu acho que ontem no mesmo horário importa" e a evidência de que importa.

## 4.3 Decomposição da série (extra)

A título ilustrativo, `seasonal_decompose` separa a série em **tendência + sazonalidade + resíduo**. Para dados horários, o período natural da sazonalidade diária é **24**. (Passar um período errado — como um valor "mensal" arbitrário — produz uma decomposição sem sentido, um erro comum.)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Período 24 = ciclo diário. Usamos ~30 dias para o gráfico ficar legível.
recorte = df["count"].loc["2011-06-01":"2011-06-30"]
resultado = seasonal_decompose(recorte, model="additive", period=24)
resultado.plot(); plt.gcf().set_size_inches(11, 6)
plt.tight_layout(); plt.show()


**Exercício 5.** Após inspecionar as sazonalidades, escreva quais features você espera que sejam mais importantes para prever a demanda — e por quê. A hora do dia certamente estará no topo; e o clima? Quanto da variação *não* explicada pelas regularidades de calendário sobra para os lags e para o modelo aprenderem?

# 5. Preparação dos Dados para Modelagem

Com as features prontas, preparamos treino, validação e teste de forma justa: definir o alvo, separar **respeitando o tempo** e organizar as matrizes `X`/`y`.

## 5.1 Definição dos problemas e dos alvos

**Regressão (forecasting de 1 passo).** Prever a contagem da **próxima hora** a partir da informação até agora. Para cada instante `t`, as features referem-se a até `t`, e o alvo é `count` em `t+1`. Construímos deslocando o alvo com `shift(-1)`.

**Classificação (evento com significado real).** Em vez de rotular artificialmente "demanda alta = acima do percentil 75" (que cria uma classe sem interpretação de negócio nem sinal para aprender), definimos um alvo **operacional**: a próxima hora vai **estourar a capacidade da frota**? Suponha que o operador consegue atender confortavelmente até um certo volume por hora; acima disso, faltam bikes e é preciso redistribuir. Fixamos esse limiar como o **percentil 70 da demanda no treino** e perguntamos se a *próxima* hora ficará acima dele.

Por que esse alvo é bom didaticamente: ele **não é uma função determinística do relógio** (ao contrário de "é horário de pico?", que o modelo só precisaria decorar a partir de `hour`). Uma hora de pico num dia frio e chuvoso pode ficar *abaixo* do limiar; um sábado de verão pode estourá-lo. O modelo precisa combinar calendário, clima e a história recente (lags) — é forecasting de verdade, não uma regra de tabela.

In [ ]:
# --- Alvo de regressão: demanda da próxima hora
df["target_reg"] = df["count"].shift(-1)

# --- Alvo de classificação: a PRÓXIMA hora estoura a capacidade da frota?
# Limiar = percentil 70 da demanda observada NO TREINO (nunca no dataset todo,
# para não vazar informação do futuro na definição do rótulo).
LIMIAR_CAPACIDADE = df.loc[:"2012-03-31", "count"].quantile(0.70)
df["target_clf"] = (df["count"].shift(-1) >= LIMIAR_CAPACIDADE).astype(int)

print(f"Limiar de capacidade (p70 do treino): {LIMIAR_CAPACIDADE:.0f} aluguéis/h")
print("Distribuição do alvo (0=dentro da capacidade, 1=estoura):")
print(df["target_clf"].value_counts(normalize=True).round(3))


O balanceamento fica em torno de 60/40 — saudável, sem ser trivial. E o rótulo depende de `count` **futuro** comparado a um limiar fixado **só com dados de treino**: se calculássemos o percentil sobre a série inteira, a própria definição do alvo já teria olhado o futuro. Detalhes assim são onde o vazamento se esconde.

## 5.2 Divisão treino / validação / teste respeitando o tempo

O erro clássico seria embaralhar e sortear treino/teste. **Não se faz isso com séries temporais:** precisamos simular o cenário real de prever o futuro, treinando no passado e testando no que veio depois. Logo, o teste é sempre o trecho **final** da série.

Como temos 2011–2012, dividimos cronologicamente:

- **Treino:** 2011-01 até 2012-03
- **Validação:** 2012-04 até 2012-08 (ajuste de modelo/hiperparâmetros)
- **Teste:** 2012-09 até 2012-12 (avaliação final, dados nunca vistos)

In [ ]:
corte_val  = "2012-04-01"
corte_test = "2012-09-01"

df = df.sort_index()
train_df = df.loc[:"2012-03-31 23:00"].copy()
val_df   = df.loc[corte_val:"2012-08-31 23:00"].copy()
test_df  = df.loc[corte_test:].copy()

print(f"Treino:    {train_df.index.min()} → {train_df.index.max()}  ({len(train_df)} h)")
print(f"Validação: {val_df.index.min()} → {val_df.index.max()}  ({len(val_df)} h)")
print(f"Teste:     {test_df.index.min()} → {test_df.index.max()}  ({len(test_df)} h)")


> **Além do split único:** para modelos robustos usa-se **validação cruzada temporal** (`TimeSeriesSplit` do scikit-learn), que gera vários cortes que avançam no tempo (*rolling origin*), às vezes com um `gap` entre treino e teste para evitar vazamento de dependências curtas. Manteremos o split único por clareza, mas o código de CV temporal aparece no Exercício 7.

## 5.3 Seleção explícita de features (evitando vazamento)

Este é o ponto onde vazamento entra sem avisar. **Nunca** selecione features por fatiamento posicional de colunas (`df.columns[:-2]`) — a ordem das colunas muda e você acaba incluindo o alvo, ou o próprio `count` atual, sem perceber.

Listamos explicitamente o que pode entrar. Em particular, **`count` atual não é feature**: no momento de prever `t+1`, usá-lo seria trapaça (ele praticamente já contém a resposta). O modelo só pode ver *lags* de `count`, nunca o valor corrente.

In [ ]:
feature_cols = [
    # calendário
    "hour", "weekday", "month", "year", "is_weekend",
    "holiday", "workingday", "weather", "season",
    # codificação cíclica / Fourier
    "hour_sin", "hour_cos", "wday_sin", "wday_cos", "ano_sin", "ano_cos",
    # meteorologia
    "temp", "humidity", "windspeed",
    # história da série (somente passado!)
    "lag1", "lag24", "lag168",
    "roll_mean_3", "roll_mean_24", "roll_std_24", "roll_max_24",
]
# Garantia explícita: nenhum alvo, nem o count atual, entre as features.
assert "count" not in feature_cols and "target_reg" not in feature_cols \
       and "target_clf" not in feature_cols
print(f"{len(feature_cols)} features selecionadas.")


## 5.4 Matrizes X e y

Montamos `X`/`y` para cada conjunto. As primeiras horas do treino têm `NaN` nos lags/rollings (não há passado); descartamos essas linhas em vez de preencher com zero. Nos conjuntos de validação e teste os lags já estão definidos, porque há histórico imediatamente antes de cada corte.

In [ ]:
def montar_xy(frame, alvo):
    cols = feature_cols + [alvo]
    limpo = frame[cols].dropna()          # remove linhas sem lag e sem alvo
    return limpo[feature_cols], limpo[alvo]

X_train, y_train_reg = montar_xy(train_df, "target_reg")
X_val,   y_val_reg   = montar_xy(val_df,   "target_reg")
X_test,  y_test_reg  = montar_xy(test_df,  "target_reg")

# Para classificação, os mesmos X servem; alinhamos o alvo pelos índices de X.
y_train_clf = train_df.loc[X_train.index, "target_clf"]
y_val_clf   = val_df.loc[X_val.index,   "target_clf"]
y_test_clf  = test_df.loc[X_test.index,  "target_clf"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Colunas conferem: {list(X_train.columns) == feature_cols}")


**Exercício 6.** Confirme as dimensões e inspecione `X_train.head()`. Verifique que nenhuma feature inválida escapou (nada de `count` atual, nada de alvo). Se tiver adicionado features no Exercício 4, inclua-as em `feature_cols` e confira que continuam disponíveis no momento da previsão.

# 6. Modelagem: Regressão (Previsão Numérica)

Vamos prever `count` da próxima hora. Um detalhe de engenharia importante: modelos lineares e o escalonamento **devem viver dentro de um `Pipeline`**, com o `StandardScaler` ajustado só no treino. Fazer o `fit` do scaler sobre treino+validação juntos é uma forma silenciosa de vazamento — o `Pipeline` elimina esse risco por construção.

## 6.1 Baseline ingênuo

Antes de qualquer modelo, um **baseline** honesto: prever que a próxima hora será igual à hora atual (*naïve forecast*, `lag1`). Todo modelo precisa vencer isso para justificar sua existência.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

pred_naive = X_val["lag1"].values
mae_naive  = mean_absolute_error(y_val_reg, pred_naive)
rmse_naive = root_mean_squared_error(y_val_reg, pred_naive)
print(f"Baseline naïve (t+1 = t) — MAE: {mae_naive:.2f} | RMSE: {rmse_naive:.2f}")


## 6.2 Regressão linear (dentro de um Pipeline)

O `Pipeline` encapsula escalonamento + modelo. O `StandardScaler` é ajustado apenas em `X_train` quando chamamos `fit`, e reaplicado em `X_val`/`X_test` sem reajustar.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

lin = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
lin.fit(X_train, y_train_reg)
pred_val_lin = lin.predict(X_val)

mae_lin  = mean_absolute_error(y_val_reg, pred_val_lin)
rmse_lin = root_mean_squared_error(y_val_reg, pred_val_lin)
r2_lin   = lin.score(X_val, y_val_reg)
print(f"Regressão Linear — MAE: {mae_lin:.2f} | RMSE: {rmse_lin:.2f} | R²: {r2_lin:.3f}")


In [ ]:
# Coeficientes (sobre features padronizadas → comparáveis entre si)
coefs = pd.Series(lin.named_steps["model"].coef_, index=feature_cols)
print("Maiores pesos (valor absoluto):")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10))


Como as features estão padronizadas, os coeficientes são diretamente comparáveis. Espera-se peso alto para `lag1`, `lag24` e para os termos de hora — coerente com a ACF. Cuidado, porém: correlação entre features (por exemplo `lag24` e `hour`) torna a leitura individual dos coeficientes imprecisa.

## 6.3 Modelos de árvore e ensemble

Modelos de árvore capturam não linearidades e interações (por exemplo, "o clima só afeta em certas horas") e não precisam de escalonamento. Testamos **Random Forest** e **Gradient Boosting (XGBoost)**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=14,
                           n_jobs=-1, random_state=42)
rf.fit(X_train, y_train_reg)
pred_val_rf = rf.predict(X_val)
print(f"Random Forest — MAE: {mean_absolute_error(y_val_reg, pred_val_rf):.2f} "
      f"| RMSE: {root_mean_squared_error(y_val_reg, pred_val_rf):.2f}")


In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train_reg,
        eval_set=[(X_val, y_val_reg)], verbose=False)
pred_val_xgb = xgb.predict(X_val)
print(f"XGBoost — MAE: {mean_absolute_error(y_val_reg, pred_val_xgb):.2f} "
      f"| RMSE: {root_mean_squared_error(y_val_reg, pred_val_xgb):.2f}")


In [ ]:
# Placar na validação
placar = pd.DataFrame({
    "modelo": ["Naïve (lag1)", "Linear", "Random Forest", "XGBoost"],
    "MAE":  [mae_naive, mae_lin,
             mean_absolute_error(y_val_reg, pred_val_rf),
             mean_absolute_error(y_val_reg, pred_val_xgb)],
    "RMSE": [rmse_naive, rmse_lin,
             root_mean_squared_error(y_val_reg, pred_val_rf),
             root_mean_squared_error(y_val_reg, pred_val_xgb)],
}).set_index("modelo").round(2)
print(placar)


Em geral, RF e XGBoost superam a regressão linear e o baseline, porque as relações são não lineares e há interações. Em competições de forecasting, modelos de gradient boosting (LightGBM/XGBoost) com boas features frequentemente batem métodos clássicos — o que justifica testá-los aqui.

**Exercício 7.** Ajuste hiperparâmetros usando **validação cruzada temporal**. O esqueleto abaixo usa `TimeSeriesSplit` como *splitter* do `GridSearchCV` — nunca o `KFold` padrão, que embaralharia o tempo.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

tscv = TimeSeriesSplit(n_splits=4)
grade = {"max_depth": [8, 14], "n_estimators": [200, 400]}
busca = GridSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=42),
    grade, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
# Descomente para rodar (pode levar alguns minutos):
# busca.fit(X_train, y_train_reg)
# print(busca.best_params_, -busca.best_score_)
print("TimeSeriesSplit pronto:", tscv)


## 6.4 Comparação com modelos clássicos (extra)

Vale conhecer os métodos estatísticos tradicionais. **ARIMA/SARIMA** modela bem sazonalidade linear e tendência, mas ajustar um SARIMA sazonal sobre ~17 mil pontos horários (período 24) é **inviável numa aula** — trava por muitos minutos. O caminho prático é **agregar para o nível diário** e usar sazonalidade semanal (`m=7`). Deixamos como exercício para casa.

In [ ]:
# EXERCÍCIO PARA CASA (não roda na aula — descomente com calma):
# !pip install pmdarima
# from pmdarima import auto_arima
# diario = df["count"].resample("D").sum().dropna()
# treino_d = diario.loc[:"2012-08-31"]
# modelo_arima = auto_arima(treino_d, seasonal=True, m=7, trace=True,
#                           suppress_warnings=True, stepwise=True)
# print(modelo_arima.summary())
print("Bloco ARIMA deixado como exercício (ver comentários).")


## 6.5 Avaliação final no conjunto de teste

Escolhido o melhor modelo pela validação, avaliamos **uma única vez** no teste — dados jamais usados. Também plotamos previsto vs. real num recorte, e reportamos o erro **excluindo horas que foram imputadas**, para não medir desempenho contra valores que nós mesmos inventamos.

In [ ]:
modelos = {"Linear": lin, "RandomForest": rf, "XGBoost": xgb}
melhor_nome = min(modelos, key=lambda k: root_mean_squared_error(
    y_val_reg, modelos[k].predict(X_val)))
melhor = modelos[melhor_nome]
print(f"Melhor na validação: {melhor_nome}")

pred_test = melhor.predict(X_test)
# máscara de horas reais (não imputadas) no teste
reais = ~test_df.loc[X_test.index, "count_imputado"].values
mae_test  = mean_absolute_error(y_test_reg[reais], pred_test[reais])
rmse_test = root_mean_squared_error(y_test_reg[reais], pred_test[reais])
print(f"TESTE ({melhor_nome}) — MAE: {mae_test:.2f} | RMSE: {rmse_test:.2f}")


In [ ]:
recorte = slice("2012-10-01", "2012-10-14")
plt.figure(figsize=(12, 4))
plt.plot(y_test_reg.loc[recorte].index, y_test_reg.loc[recorte].values,
         label="Real", lw=1.2)
serie_pred = pd.Series(pred_test, index=X_test.index).loc[recorte]
plt.plot(serie_pred.index, serie_pred.values, label="Previsto", lw=1.2, alpha=0.8)
plt.title(f"Demanda real vs. prevista no teste — {melhor_nome} (2 semanas de out/2012)")
plt.xlabel("Data"); plt.ylabel("Aluguéis/hora"); plt.legend()
plt.tight_layout(); plt.show()


Repare onde o modelo erra mais: picos e vales são os instantes mais difíceis. Num caso de negócio, decidiríamos se o erro é aceitável — um RMSE de 40 bikes/h pode ser ótimo se a média é ~190/h, mas ruim se fosse ~50/h. O contexto define o critério.

# 7. Modelagem: Classificação (Prever Estouro de Capacidade)

Com as **mesmas features**, treinamos classificadores para prever se a próxima hora estoura a capacidade da frota. Muda o alvo e o algoritmo; o pipeline é análogo ao da regressão.

## 7.1 Treinamento

Usamos **Regressão Logística** (linear, dentro de um `Pipeline` com escalonamento) e **Random Forest Classifier** (não linear). Mantemos `class_weight="balanced"` para não favorecer a classe majoritária.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
logreg.fit(X_train, y_train_clf)
proba_val_log = logreg.predict_proba(X_val)[:, 1]
pred_val_log  = logreg.predict(X_val)

rf_clf = RandomForestClassifier(n_estimators=200, max_depth=14,
                                class_weight="balanced",
                                n_jobs=-1, random_state=42)
rf_clf.fit(X_train, y_train_clf)
proba_val_rf = rf_clf.predict_proba(X_val)[:, 1]
pred_val_rf  = rf_clf.predict(X_val)
print("Classificadores treinados.")


## 7.2 Avaliação

Para classificação, olhamos **acurácia, F1 e ROC AUC**. Como as classes não estão perfeitamente equilibradas, a acurácia sozinha pode enganar — F1 e AUC contam a história real, sobretudo para a classe "estoura capacidade", que é a que interessa operacionalmente.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix,
                             classification_report)

for nome, pred, proba in [("Logística", pred_val_log, proba_val_log),
                          ("Random Forest", pred_val_rf, proba_val_rf)]:
    print(f"\n=== {nome} ===")
    print(f"Acurácia: {accuracy_score(y_val_clf, pred):.3f} | "
          f"F1: {f1_score(y_val_clf, pred):.3f} | "
          f"ROC AUC: {roc_auc_score(y_val_clf, proba):.3f}")
print("\nMatriz de confusão (RF):\n", confusion_matrix(y_val_clf, pred_val_rf))


In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_val_clf, proba_val_rf, ax=ax1, name="RF")
ax1.plot([0,1],[0,1],"--",color="gray"); ax1.set_title("Curva ROC")
PrecisionRecallDisplay.from_predictions(y_val_clf, proba_val_rf, ax=ax2, name="RF")
ax2.set_title("Curva Precisão–Revocação")
plt.tight_layout(); plt.show()


## 7.3 Importância das features

In [ ]:
importancias = (pd.Series(rf_clf.feature_importances_, index=feature_cols)
                  .sort_values(ascending=False))
plt.figure(figsize=(9, 4))
importancias.head(12).plot.barh()
plt.gca().invert_yaxis()
plt.title("Top 12 features para prever estouro de capacidade (Random Forest)")
plt.xlabel("Importância"); plt.tight_layout(); plt.show()


No topo devem aparecer os **lags** (`lag24`, `lag1`) e os termos de hora — o modelo aprende que a demanda futura se parece com a de ontem no mesmo horário e com a hora recente. Ao contrário de um alvo "horário de pico", aqui `hour` sozinho não resolve: o modelo precisa combinar história recente, calendário e clima para acertar quando a demanda cruza o limiar.

**Exercício 8.** Analise os erros: filtre os falsos negativos (a hora estourou a capacidade e o modelo não previu). Concentram-se em algum contexto — clima ruim, transições de estação, feriados? Esses casos costumam inspirar novas features.

## 7.4 Avaliação final no teste (classificação)

In [ ]:
melhor_clf = rf_clf   # normalmente o RF vence
pred_test_clf  = melhor_clf.predict(X_test)
proba_test_clf = melhor_clf.predict_proba(X_test)[:, 1]

print(f"TESTE — Acurácia: {accuracy_score(y_test_clf, pred_test_clf):.3f} | "
      f"F1: {f1_score(y_test_clf, pred_test_clf):.3f} | "
      f"ROC AUC: {roc_auc_score(y_test_clf, proba_test_clf):.3f}")
print("\n", classification_report(y_test_clf, pred_test_clf,
                                   target_names=["dentro", "estoura"]))


# 8. Do Modelo ao Endpoint: Serialização

Fecha o ciclo da pós de Desenvolvimento Web: como levar o modelo treinado para produção. Salvamos o objeto **inteiro** com `joblib` — no caso da regressão, o `Pipeline` já carrega o scaler junto, então em produção basta chamar `predict` com as mesmas features, sem reescalonar à mão.

In [ ]:
import joblib

joblib.dump(melhor, "modelo_forecast_bike.joblib")
joblib.dump(feature_cols, "feature_cols.joblib")
print(f"Modelo de forecast salvo: {melhor_nome}")

# Simulação de uso em produção: recarregar e prever uma hora
modelo_prod = joblib.load("modelo_forecast_bike.joblib")
cols_prod   = joblib.load("feature_cols.joblib")
exemplo = X_test.iloc[[-1]][cols_prod]
print(f"Previsão para a próxima hora: {modelo_prod.predict(exemplo)[0]:.0f} aluguéis")


**Esqueleto de um endpoint (pseudocódigo).** Num serviço web, a requisição traz o timestamp; o servidor calcula as mesmas features (lags a partir do histórico no banco, componentes de calendário a partir do relógio) e chama `predict`:

```python
# from fastapi import FastAPI
# app = FastAPI()
# modelo = joblib.load("modelo_forecast_bike.joblib")
# cols   = joblib.load("feature_cols.joblib")
#
# @app.post("/prever")
# def prever(payload: dict):
#     X = montar_features(payload)   # MESMA lógica de feature engineering do treino
#     return {"aluguveis_previstos": float(modelo.predict(X[cols])[0])}
```

> **A regra de ouro em produção:** as features na inferência têm de ser calculadas **exatamente** como no treino. Reaproveite a mesma função de feature engineering nos dois lugares — divergências aqui são a causa nº 1 de modelos que vão bem no teste e mal em produção (*training–serving skew*).

# 9. Conclusões e Próximos Passos

O que percorremos, e os pontos que valem levar adiante:

- **Continuidade temporal primeiro.** Reindexar para uma grade horária completa foi o que tornou lags e janelas móveis temporalmente corretos. Sem isso, as features mais poderosas viram ruído silencioso.
- **Look-ahead bias é o inimigo recorrente.** Ele aparece em janelas móveis (resolvido com `.shift(1)`), na seleção de features (lista explícita, sem `count` atual), no escalonamento (resolvido pelo `Pipeline`) e no split (corte cronológico, nunca embaralhado).
- **Alvos com significado.** Trocamos o rótulo artificial por "estoura a capacidade da frota" — leitura de negócio direta, balanceamento saudável e, crucialmente, um alvo que exige *aprender* (não decorar o relógio).
- **Baseline honesto.** O naïve `t+1 = t` é a régua mínima; um modelo só se justifica se o superar.
- **Do notebook ao serviço.** O `Pipeline` serializado leva o pré-processamento junto e encaixa direto atrás de um endpoint.

**Para aprofundar:** previsão de múltiplos passos à frente (recursiva ou multi-output); incorporar variáveis exógenas (feriados especiais, eventos, clima previsto); testar LightGBM e modelos de deep learning (LSTM/Temporal Fusion Transformer) quando houver dados e tempo; e validação cruzada temporal completa para estimativas de erro mais confiáveis.